# Local Backend

evy's default backend is Google Earth Engine (`backend="gee"`), which runs computations on Google's servers. This recipe demonstrates the **local backend** (`backend="local"`), which downloads MODIS rasters from Microsoft's Planetary Computer and computes statistics on your machine.

**When to use the local backend:**
- You don't have a GEE account or can't authenticate
- You're working with small regions where download + local compute is fast enough
- You need access to the raw rasters (e.g., for custom masking or debugging)

**Trade-offs:**
- Local is **slower** for large countries or long date ranges (downloads full tiles)
- Local is **faster** for very small regions (no round-trip to GEE servers)
- Local uses ESA WorldCover for cropland masking (class 40), while GEE uses Dynamic World (class 4)
- No authentication required for the default Planetary Computer path

See [Troubleshooting](../../docs/troubleshooting.md) if you hit HDF4 driver errors or memory issues.

In [ ]:
# papermill parameters
country = "RWA"
admin_level = 1
start_date = "2023-01-01"
end_date = "2023-12-31"
quick_mode = False


In [ ]:
if quick_mode:
    end_date = "2023-02-28"


In [ ]:
import evy
import numpy as np

## One-line local computation

The simplest switch: pass `backend="local"` to `zonal_stats`. Everything else stays the same.

The output has the same columns as the GEE backend (`date`, `shapeName`, `mean`, ...), so downstream code works with either backend.

In [ ]:
gdf = evy.get_boundaries(country, admin_level=admin_level)
df_local = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    backend="local",
    source="modis",
    start_date=start_date,
    end_date=end_date,
    freq=evy.MONTHLY,
    stats=["mean", "std"],
)
df_local.head()

In [ ]:
evy.plot_time_series(df_local, value_col="mean", title="EVI (Local Backend)")

## The low-level pipeline

For full control, call the three steps separately:

1. **`load_modis`** — STAC search → lazy `xarray.Dataset` with `evi_raw` and `qa` bands
2. **`load_landcover`** — aligned ESA WorldCover raster
3. **`compute_zonal_stats`** — quality mask → scale → temporal aggregation → cropland mask → extraction

This is equivalent to `zonal_stats(backend="local")` but lets you inspect intermediate rasters.

In [ ]:
ds = evy.load_modis(gdf, start_date=start_date, end_date=end_date)
print(ds)

In [ ]:
lc = evy.load_landcover(gdf, ds)
print(f"Land cover shape: {lc.shape}")
values = lc.values.flatten()
valid = values[~np.isnan(values)].astype(int)
print(f"Unique classes: {np.unique(valid)}")

In [ ]:
df_pipeline = evy.compute_zonal_stats(
    ds,
    gdf,
    zone_col="shapeName",
    land_cover=lc,
    freq=evy.MONTHLY,
    stats=["mean", "median"],
)
df_pipeline.head()

## NASA Earthdata backend

If you need MODIS data not available on Planetary Computer, evy also supports NASA Earthdata via the `earthaccess` library. This requires authentication:

```python
import earthaccess
earthaccess.login(persist=True)
```

Once authenticated, the `earthaccess` backend streams MODIS bands via OPeNDAP (as NetCDF4), which avoids the HDF4 driver issues common on macOS. See [Troubleshooting](../../docs/troubleshooting.md) for details.

The Planetary Computer path is the recommended default — it requires no authentication and serves Cloud-Optimized GeoTIFFs for fast partial reads.

## Output column naming

Both backends return the same columns: `date` (start of each period), your `zone_col`, one column per statistic (`mean`, `median`, `min`, `max`, `std`, `sum`, `count`), and `geometry` when `include_geometry=True`. You can pass output from either backend to `calculate_phenology` or the plotting functions without changes.